# 10 — Experiment C3: Enriched GRU (structural features, TextSkill slot left open)

Follow-up to
`05_gru_neural_elo.ipynb` (Exp C), `08_gru_elo.ipynb` (Exp C2), `07_xgboost_trajectory.ipynb`
(Exp A'/B'), `06_confidence_intervals_cv.ipynb`. Exp C2 (GRU + ELO-at-head) is the strongest
model built so far (test Brier 0.16088) but its edge over Exp C (ELO-free GRU) and Exp B'
(ELO-included XGB + trajectory) was **not** statistically significant on the single-split paired
bootstrap.

**This notebook asks: do additional structural features — beyond a bare ELO scalar — give the
GRU a real, defensible edge?** Everything here is built on top of the *exact* Exp C2 pipeline
(same raw data, same duplicate-and-flip rows, same 80/20 time split, same architecture family,
same hyperparameters/seed, same calibration methodology) — only the model's input surface is
enriched:

**Per-step sequence, extended 7 → 9 dims:**
- `OpponentEloBefore` (NEW) — the opponent's own point-in-time `EloBefore` in each historical
  match, reconstructed directly from `matches_export.csv`'s `WinnerEloBefore`/`LoserEloBefore`
  columns (already present in the raw export — no re-derivation needed). Scaled with the same
  TRAIN-only `ELO_MEAN`/`ELO_STD` used for the head-level ELO scalars (Exp C2 Step 7 discipline).
- `WasH2H` (NEW) — whether the player had already faced that exact opponent before that
  historical match's date (binary flag). Vectorized
  `groupby(['IgracId','OpponentId']).cumcount() > 0`, strictly point-in-time by construction (the
  same `(IgracId, Datum, SusretId)` global sort as every other point-in-time column in this
  project guarantees the subgroup order is also chronological).

**Head — per player (×2), concatenated alongside Exp C2's `elo_p_scaled`/`elo_o_scaled`:**
- `CareerWinRateBefore`, `FormaLast10`, `MatchCountBefore`, `Age` — reused directly from
  `features.parquet`'s existing point-in-time columns (`player_career_winrate_before`,
  `player_forma_last10`, `player_match_count_before`, `player_age`, and the `opponent_*`
  equivalents). These already exist and are already leakage-audited in
  `02_feature_engineering.ipynb` — this notebook does not recompute them, only reuses and feeds
  them to the GRU head for the first time (Exp C/C2 never saw them; only XGBoost did).
- **`TextSkillScore` — SLOT LEFT OPEN.** A concurrent process is scoring `Trenutci` (free-text
  player self-descriptions) via Gemini; that work is explicitly out of scope for this notebook and
  its output files are not read here. Two head columns (`text_p`, `text_o`) are wired into the
  model's input tensor NOW, filled with a **neutral constant (`0.0`, after the same
  standardization convention as the other head scalars — i.e. "no information")**, so plugging in
  the real per-player score later is a one-line change (replace the constant-fill with a merged
  column) + a retrain — not an architecture change. See the code comment at the fill site.

**Head — per matchup (×1):**
- `h2h_win_rate`, `h2h_total` — reused directly from `features.parquet`
  (`player_h2h_win_rate_before`, `h2h_total_before`) — first time exposed to the GRU (Exp C/C2
  never saw H2H at all).
- `h2h_sets_margin` (NEW — **a previously-planned-but-not-yet-built feature, built here**) — for each
  match, the sum over the pair's STRICTLY PRIOR H2H meetings of (this player's sets won −
  opponent's sets won). Built vectorized from the per-step `SetMargin` column already computed
  for the GRU sequence (`groupby(['IgracId','OpponentId'])['SetMargin'].cumsum().shift(...)`),
  looked up onto `sym` via the exact same `player_pos` index array already used to resolve GRU
  history positions — no new merge needed. Leakage-audited explicitly below (Step 5).

**Reused, unchanged from Exp C/C2:** `matches_export.csv` + `features.parquet`'s `Split` column
(train 2017-05-08..2025-01-18 / test 2025-01-19..2026-07-08, 3,426 test matches / 6,852
duplicate-and-flip rows), duplicate-and-flip symmetric expansion, `calib_frac=0.15` /
`early_stop_frac=0.10` chronological TRAIN-only carve, `TwoTowerGRU`-family architecture
(`hidden_size=24`, `dropout=0.3`, AdamW `lr=2e-3` / `weight_decay=1e-4`, `BATCH_SIZE=512`,
`MAX_EPOCHS=60`, `PATIENCE=8`, seed=42), temperature-vs-isotonic calibration on the `calib` slice,
evaluation on the identical 6,852-row TEST set plus match-level bootstrap CIs (2,000 resamples,
seed 42) and paired-bootstrap Brier differences. **No walk-forward CV in this pass** (single split
+ bootstrap only, per this notebook's scope).

**Runtime-safety note:** every new feature below was engineered with vectorized pandas
(`groupby`/`shift`/`cumcount`/`cumsum` — no per-row Python loops) and the full pipeline + one GRU
train was validated end-to-end in a standalone script (`scripts/exp_c3_validate.py`, ~98s total)
*before* this notebook was assembled, per this project's runtime-safety discipline.


In [ ]:
import json
import pickle
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.use_deterministic_algorithms(False)  # GRU has no deterministic CPU kernel path for this op set

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

DATA_DIR = Path('../data')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device('cpu')
print(f'PyTorch {torch.__version__}, device={device}, threads={torch.get_num_threads()}')


## Step 1 — Load raw export + the P2 engineered frame (identical to Exp C/C2)

`features.parquet` supplies `PlayerId`/`OpponentId`/`Label`/`Split`/`Datum`/`SusretId` (the
duplicate-and-flip row set and time split), `player_elo`/`opponent_elo` (Exp C2's head ELO), and
now also — reused for the first time by a GRU — `player_career_winrate_before`,
`player_forma_last10`, `player_match_count_before`, `player_age`, `player_h2h_win_rate_before`,
`h2h_total_before` (and `opponent_*` equivalents).

In [ ]:
raw = pd.read_csv(DATA_DIR / 'matches_export.csv', parse_dates=['Datum'])
raw['Datum'] = pd.to_datetime(raw['Datum']).dt.tz_localize(None)
raw = raw.sort_values(['Datum', 'SusretId']).reset_index(drop=True)

sym = pd.read_parquet(DATA_DIR / 'features.parquet')
with open(DATA_DIR / 'feature_tags.json', encoding='utf-8') as f:
    TAGS = json.load(f)

print(f'raw matches: {raw.shape}')
print(f'sym (duplicate-and-flip) frame: {sym.shape}')
print(f"Split date (from P2): {TAGS['split_date']}, train matches: {TAGS['n_matches_train']:,}, "
      f"test matches: {TAGS['n_matches_test']:,}")
sym[['SusretId', 'Datum', 'PlayerId', 'OpponentId', 'Label', 'Perspective', 'Split']].head()


## Step 2 — Long-format player-match log, extended with `OppEloBefore` (NEW)

Same construction as `05`/`08` Step 2, with two extra columns pulled directly from the raw export
(no re-derivation): `MyEloBefore` (this player's own point-in-time ELO in that historical match —
not used as a per-step input in this notebook, kept only for completeness/consistency) and
`OppEloBefore` (**the opponent's** point-in-time ELO in that same historical match — the new
per-step feature).

In [ ]:
# Cells 5/7/9/11 (Steps 2-5) now delegate the long-format point-in-time
# player log construction to the SHARED, notebook-free
# `serving/feature_pipeline.py::build_player_log` module -- imported here
# instead of defined inline (the project's #1 anti training/serving skew
# measure: reuse the exact production feature logic, don't reimplement it).
# Plumbing-only change -- identical computation, identical numeric result
# (verified in a regression test against this exact notebook's own
# historical rows: 0.00e+00 max diff).
import sys
sys.path.insert(0, '..')  # repo root (notebooks -> repo root), so `serving.*` resolves
from serving.feature_pipeline import build_player_log

player_log = build_player_log(raw)
print(f'player_log: {player_log.shape}')
player_log.head(3)


## Step 3 — Per-step point-in-time features (identical to Exp C/C2 — still ELO-free per step, except the new `OppEloBefore`)

`Won`, `SetMargin`, `DaysSinceLastMatch`, `CareerWinRateBefore`, `HomeShareBefore` — unchanged from
`05`/`08`. Cross-checked against the SQL-exported `MatchCountBefore` exactly as before.

In [ ]:
# Steps 3's MatchCountBefore/HomeShareBefore/DaysSinceLastMatch/CareerWinRateBefore/
# SetMargin are now all computed inside build_player_log() above (Cell 5).
# This cell keeps ONLY the notebook-only QA step: cross-checking the
# pandas-computed MatchCountBefore against the SQL-exported ground truth
# (WinnerMatchCountBefore/LoserMatchCountBefore) -- this assert is what
# originally proved the point-in-time pandas ordering is correct, and it is
# NOT part of feature_pipeline.py (there is no SQL ground-truth column to
# check against at serving time -- serving has no "expected" precomputed
# count to compare to).
check_w = raw[['SusretId', 'WinnerId', 'WinnerMatchCountBefore']].rename(columns={'WinnerId': 'IgracId', 'WinnerMatchCountBefore': 'Exp'})
check_l = raw[['SusretId', 'LoserId', 'LoserMatchCountBefore']].rename(columns={'LoserId': 'IgracId', 'LoserMatchCountBefore': 'Exp'})
chk = pd.concat([check_w, check_l])
mm = player_log.merge(chk, on=['SusretId', 'IgracId'], how='left')
mismatches = (mm['MatchCountBefore'] != mm['Exp']).sum()
assert mismatches == 0, 'pandas point-in-time ordering does not match the SQL export!'
print(f'MatchCountBefore cross-check vs SQL export: {mismatches} mismatches -- OK.')
player_log[['Won', 'SetMargin', 'DaysSinceLastMatch', 'LigaKoeficjent', 'HomeShareBefore', 'CareerWinRateBefore']].describe()


## Step 4 — Opponent-quality-at-that-time proxy (identical to Exp C/C2)

Self-merge on `(SusretId, OpponentId)` — ELO-free by construction (reused unchanged).

In [ ]:
# Step 4's opponent-quality-at-that-time self-merge is now inside
# build_player_log() above (Cell 5) -- this cell keeps only the diagnostic
# output.
print(f"OpponentCareerWinRateBefore NaN share (opponent's own debut match): "
      f"{player_log['OpponentCareerWinRateBefore'].isna().mean():.2%}")
player_log[['OpponentCareerWinRateBefore']].describe()


## Step 5 — NEW: per-step H2H features (`WasH2H`) + the dropped `h2h_sets_margin` feature

Both vectorized (`groupby`/`cumcount`/`cumsum` — no per-row Python loops), both strictly
point-in-time by the same `(IgracId, Datum, SusretId)` global-sort argument used everywhere else
in this project (a subsequence of a chronologically sorted sequence, filtered by a fixed
`OpponentId`, is still chronologically sorted).

- **`H2HMeetingsBefore`** — how many times this player had already faced this exact opponent
  strictly before this row's match (`groupby(['IgracId','OpponentId']).cumcount()`).
  **`WasH2H`** (the actual per-step GRU input) is the binarized flag (`> 0`) — the original design
  offered either "1/0, or prior-meeting count"; binary was chosen for simplicity (the raw count is
  kept in `player_log` for the leakage audit and is available if a future pass wants to swap it in).
- **`H2HSetsMarginBefore`** — cumulative sum of this player's own `SetMargin` against this specific
  opponent, computed through each row and then shifted by one position within the
  `(IgracId, OpponentId)` subgroup — i.e. the sum over STRICTLY PRIOR meetings only (current row's
  own `SetMargin` is excluded). This is **a previously-planned-but-not-yet-built `h2h_sets_margin`
  feature** — never implemented before this notebook (absent from `feature_tags.json` /
  `features.parquet`). Resolved onto `sym`
  further below via the exact same `player_pos` index array already used for GRU history lookups —
  no new merge required, since `player_log` rows are already 1:1 with `(IgracId, SusretId)` pairs.

### Leakage spot-check (mandatory — reapplied here for the two new columns)

In [ ]:
# Step 5's WasH2H/H2HMeetingsBefore/H2HSetsMarginBefore are now computed
# inside build_player_log() above (Cell 5) -- this cell keeps only the
# mandatory leakage spot-check (notebook-only QA, re-verifying the shared
# module's output independently on a real sample pair).
sample_pair = player_log.loc[player_log['H2HMeetingsBefore'] >= 2].iloc[0]
pair_hist = player_log[
    (player_log['IgracId'] == sample_pair['IgracId'])
    & (player_log['OpponentId'] == sample_pair['OpponentId'])
    & (player_log['Datum'] < sample_pair['Datum'])
]
assert len(pair_hist) == sample_pair['H2HMeetingsBefore'], 'WasH2H/H2HMeetingsBefore leakage!'
expected_margin = pair_hist['SetMargin'].sum()
assert abs(expected_margin - sample_pair['H2HSetsMarginBefore']) < 1e-6, 'H2HSetsMarginBefore leakage/mismatch!'
print(f"OK -- H2H leakage spot-check: IgracId={sample_pair['IgracId']} vs OpponentId={sample_pair['OpponentId']}: "
      f"{len(pair_hist)} strictly-prior meetings independently recomputed, sets margin "
      f"{expected_margin:+.1f} == {sample_pair['H2HSetsMarginBefore']:+.1f} (column value). No leakage.")

print(f"\nWasH2H positive share (has faced this exact opponent before): {player_log['WasH2H'].mean():.2%}")
player_log[['H2HMeetingsBefore', 'H2HSetsMarginBefore']].describe()


## Step 6 — Build the 9-dim per-step feature array + history precomputation

Sequence length capped at `MAX_LEN=24` (unchanged from Exp C/C2). Each real step is now a
**9-dim** vector: `[Won, SetMarginScaled, DaysRestScaled, LigaKoeficjentScaled, HomeShareBefore,
OpponentCareerWinRateBefore, OppEloBeforeScaled (NEW), WasH2H (NEW), IsPlaceholder]` — the first 6
dims and the placeholder mechanism are byte-for-byte identical to Exp C/C2; the 2 new dims are
inserted before `IsPlaceholder`.

`LIGAKOEF_SCALE` is computed TRAIN-only (the hygiene fix established in `05` Step 15, reused
unchanged). `ELO_MEAN`/`ELO_STD` are computed TRAIN-only from `sym`'s `player_elo`/`opponent_elo`
— the *exact* same constants used for Exp C2's head-level ELO scaling, reused here to also scale
the new per-step `OppEloBefore` (same underlying quantity/distribution, so sharing the constant is
correct, not just convenient).

In [ ]:
# The 9-dim per-step feature array + history precomputation now delegates to
# the SHARED feature_pipeline.py module (build_step_features / placeholder_vector
# / build_indices / build_sequence_at) instead of the original inline
# implementation -- plumbing-only change, byte-identical numeric result
# (verified: 0.00e+00 max diff against this exact construction on real
# historical rows, incl. a debut-match length=1 case and a full MAX_LEN=24 case).
from serving.feature_pipeline import (
    MAX_LEN, N_FEATS, placeholder_vector, build_step_features, build_indices, build_sequence_at,
)

SETMARGIN_SCALE = 12.0   # fixed domain constant, unchanged from Exp C/C2
DAYSREST_CLIP = 90.0     # fixed domain constant, unchanged from Exp C/C2

_split_date = pd.Timestamp(TAGS['split_date'])
_train_mask_pl = player_log['Datum'] < _split_date
LIGAKOEF_SCALE = float(player_log.loc[_train_mask_pl, 'LigaKoeficjent'].max())  # train-only max (05 Step 15 fix)
print(f'LIGAKOEF_SCALE (train-only) = {LIGAKOEF_SCALE}')

# ELO scaling -- TRAIN-ONLY mean/std pooled over player_elo/opponent_elo from `sym`, identical
# source/discipline to Exp C2 Step 7. Reused for BOTH the head-level ELO scalars (Step 8 below)
# and this per-step OppEloBefore scaling -- same underlying quantity, same distribution.
train_elo_mask = sym['Split'] == 'train'
ELO_MEAN = float(pd.concat([sym.loc[train_elo_mask, 'player_elo'], sym.loc[train_elo_mask, 'opponent_elo']]).mean())
ELO_STD = float(pd.concat([sym.loc[train_elo_mask, 'player_elo'], sym.loc[train_elo_mask, 'opponent_elo']]).std())
print(f'ELO_MEAN (train-only) = {ELO_MEAN:.2f}, ELO_STD (train-only) = {ELO_STD:.2f}')

SCALING = {
    'setmargin_scale': SETMARGIN_SCALE, 'daysrest_clip': DAYSREST_CLIP,
    'ligakoef_scale': LIGAKOEF_SCALE, 'elo_mean': ELO_MEAN, 'elo_std': ELO_STD,
}
step_feats = build_step_features(player_log, SCALING)
PLACEHOLDER_VEC = placeholder_vector()

player_log, first_idx, count_idx, idx_lookup = build_indices(player_log)

def build_history(igrac_id, susret_id):
    '''(seq[MAX_LEN, N_FEATS] float32, length) -- history STRICTLY before this match.'''
    gpos = idx_lookup[(igrac_id, susret_id)]
    return build_sequence_at(step_feats, first_idx, igrac_id, gpos, max_len=MAX_LEN)

n_rows = len(player_log)
HIST_ARRAY = np.zeros((n_rows, MAX_LEN, N_FEATS), dtype=np.float32)
LEN_ARRAY = np.zeros(n_rows, dtype=np.int64)
igrac_arr, susret_arr = player_log['IgracId'].values, player_log['SusretId'].values
t_hist = time.time()
for i in range(n_rows):
    HIST_ARRAY[i], LEN_ARRAY[i] = build_history(igrac_arr[i], susret_arr[i])
print(f'Precomputed histories for {n_rows:,} player-match rows in {time.time()-t_hist:.2f}s.')
print(f'Sequence length -- min={LEN_ARRAY.min()}, median={np.median(LEN_ARRAY):.0f}, max={LEN_ARRAY.max()}')


### Leakage spot-check (mandatory — Part 2.2/2.2.1 discipline, reapplied here)

In [ ]:
p_check = (player_log['_gpos'] - player_log['IgracId'].map(first_idx))
assert (p_check == player_log['MatchCountBefore']).all(), 'group contiguity assumption violated!'
print('OK -- p (derived from global sort) == MatchCountBefore for all rows (group contiguity holds).')

test_period_start = pd.Timestamp(TAGS['split_date'])
sample = player_log.loc[player_log['Datum'] >= test_period_start].iloc[100]
gpos, start = sample['_gpos'], first_idx[sample['IgracId']]
hist_rows = player_log.iloc[start:gpos]
print(f"\nSpot check -- IgracId={sample['IgracId']}, SusretId={sample['SusretId']}, "
      f"Datum={sample['Datum'].date()}, MatchCountBefore={sample['MatchCountBefore']}")
print(f'History rows found: {len(hist_rows)} (expected {sample["MatchCountBefore"]})')
assert (hist_rows['Datum'] < sample['Datum']).all(), 'LEAKAGE: a history row is not strictly prior!'
print('OK -- every history row is strictly before the predicted match date. No leakage.')


## Step 7 — Resolve `features.parquet` rows onto history positions + chronological carve

Identical to Exp C/C2: `core` (fit) / `eval` (early-stopping) / `calib` (post-hoc calibration) /
`test`, same `calib_frac=0.15`, `early_stop_frac=0.10` carve at MATCH level.

In [ ]:
sym = sym.sort_values(['Datum', 'SusretId', 'Perspective'], kind='mergesort').reset_index(drop=True)
player_pos = np.array([idx_lookup[(pid, sid)] for pid, sid in zip(sym['PlayerId'], sym['SusretId'])])
opp_pos = np.array([idx_lookup[(oid, sid)] for oid, sid in zip(sym['OpponentId'], sym['SusretId'])])
print(f'Resolved player/opponent history positions for {len(sym):,} rows.')

train_matches = (
    sym.loc[sym['Split'] == 'train', ['SusretId', 'Datum']]
    .drop_duplicates().sort_values(['Datum', 'SusretId'], kind='mergesort').reset_index(drop=True)
)
n_train_matches = len(train_matches)
CALIB_FRAC, EARLY_STOP_FRAC = 0.15, 0.10
n_calib = int(round(n_train_matches * CALIB_FRAC))
fit_order = train_matches.iloc[: n_train_matches - n_calib].reset_index(drop=True)
calib_ids = set(train_matches.iloc[n_train_matches - n_calib:]['SusretId'])
n_fit = len(fit_order)
n_eval = int(round(n_fit * EARLY_STOP_FRAC))
core_ids = set(fit_order.iloc[: n_fit - n_eval]['SusretId'])
eval_ids = set(fit_order.iloc[n_fit - n_eval:]['SusretId'])

sym['Bucket'] = np.where(sym['Split'] == 'test', 'test',
                  np.where(sym['SusretId'].isin(calib_ids), 'calib',
                  np.where(sym['SusretId'].isin(eval_ids), 'eval', 'core')))
print(sym['Bucket'].value_counts())
assert sym.loc[sym['Bucket'] == 'test'].shape[0] == TAGS['n_matches_test'] * 2


## Step 8 — NEW: head-level scalar blocks

**Per-player (×2):** ELO (Exp C2's `elo_p_scaled`/`elo_o_scaled`, unchanged), `CareerWinRateBefore`
/ `FormaLast10` (already bounded `[0,1]`, reused raw — NaN, ~2.3% on debut matches, imputed at the
neutral `0.5`, same convention as the per-step `OpponentCareerWinRateBefore`), `MatchCountBefore`
(fixed domain-constant scale `/100`, clipped — not data-derived, same style as
`SETMARGIN_SCALE`/`DAYSREST_CLIP`), `Age` (TRAIN-only mean/std, ~7.8% NaN from missing `Godiste`
imputed at the train mean i.e. `0` after centering — same train/test hygiene discipline as
`ELO_MEAN`/`ELO_STD`), and the **`TextSkillScore` slot** (neutral constant `0.0` — see note below).

**Per-matchup (×1):** `h2h_win_rate` (`player_h2h_win_rate_before`, already bounded ~`[0.05,0.95]`
via a shrinkage prior from Phase P2, no NaN, reused raw), `h2h_total` (`h2h_total_before`, fixed
domain-constant scale `/10`, clipped), and **`h2h_sets_margin`** (NEW — built in Step 5 above,
resolved onto `sym` via the `player_pos` index array, fixed domain-constant scale `/20`, clipped).

**TextSkill slot — how to wire in real scores later (one-line change):** replace the two
`sym['text_p'] = TEXT_SKILL_NEUTRAL` / `sym['text_o'] = TEXT_SKILL_NEUTRAL` lines below with a
merge of the real per-player score onto `sym` by `PlayerId`/`OpponentId` (e.g.
`sym['text_p'] = sym['PlayerId'].map(text_skill_lookup).fillna(TEXT_SKILL_NEUTRAL)`), then rerun
from Step 9 onward. `N_HEAD`, the model's input width, and every downstream cell already treat
`text_p`/`text_o` as ordinary head columns — no architecture change needed.

In [ ]:
sym['elo_p_scaled'] = (sym['player_elo'] - ELO_MEAN) / ELO_STD
sym['elo_o_scaled'] = (sym['opponent_elo'] - ELO_MEAN) / ELO_STD

# CareerWinRateBefore / FormaLast10 -- already bounded [0,1], reuse raw, NaN->0.5 neutral.
sym['cwr_p'] = sym['player_career_winrate_before'].fillna(0.5)
sym['cwr_o'] = sym['opponent_career_winrate_before'].fillna(0.5)
sym['forma_p'] = sym['player_forma_last10'].fillna(0.5)
sym['forma_o'] = sym['opponent_forma_last10'].fillna(0.5)

# MatchCountBefore -- fixed domain-constant scale (observed max 300, median 37).
MATCHCOUNT_SCALE = 100.0
sym['mc_p'] = np.clip(sym['player_match_count_before'] / MATCHCOUNT_SCALE, 0, 4)
sym['mc_o'] = np.clip(sym['opponent_match_count_before'] / MATCHCOUNT_SCALE, 0, 4)

# Age -- TRAIN-ONLY mean/std (pooled); NaN (~7.8%, missing Godiste) imputed at the train mean
# (i.e. 0 after centering) -- same train/test hygiene discipline as ELO_MEAN/ELO_STD.
train_age_mask = sym['Split'] == 'train'
AGE_MEAN = float(pd.concat([sym.loc[train_age_mask, 'player_age'], sym.loc[train_age_mask, 'opponent_age']]).mean())
AGE_STD = float(pd.concat([sym.loc[train_age_mask, 'player_age'], sym.loc[train_age_mask, 'opponent_age']]).std())
sym['age_p'] = ((sym['player_age'] - AGE_MEAN) / AGE_STD).fillna(0.0)
sym['age_o'] = ((sym['opponent_age'] - AGE_MEAN) / AGE_STD).fillna(0.0)
print(f'AGE_MEAN (train-only) = {AGE_MEAN:.1f}, AGE_STD (train-only) = {AGE_STD:.1f}')

# --- TextSkillScore SLOT (concurrent Gemini-scoring agent -- not merged in this pass) ---
# Neutral constant fill for THIS run ("C3 without text"). See Step 8 markdown above for the
# one-line swap needed to wire in real per-player scores later.
TEXT_SKILL_NEUTRAL = 0.0
sym['text_p'] = TEXT_SKILL_NEUTRAL
sym['text_o'] = TEXT_SKILL_NEUTRAL

# Per-matchup: h2h_win_rate / h2h_total reused raw+scaled; h2h_sets_margin NEW (built in Step 5).
H2HTOTAL_SCALE = 10.0        # fixed domain constant (observed max 18, most mass 0-3)
H2H_SETSMARGIN_SCALE = 20.0  # fixed domain constant (per-match margin capped ~36, few H2H meetings typically)
sym['h2h_wr'] = sym['player_h2h_win_rate_before']
sym['h2h_total_scaled'] = np.clip(sym['h2h_total_before'] / H2HTOTAL_SCALE, 0, 3)
sym['h2h_sets_margin_scaled'] = np.clip(player_log['H2HSetsMarginBefore'].values[player_pos] / H2H_SETSMARGIN_SCALE, -3, 3)

HEAD_COLS = ['elo_p_scaled', 'elo_o_scaled', 'cwr_p', 'cwr_o', 'forma_p', 'forma_o',
             'mc_p', 'mc_o', 'age_p', 'age_o', 'text_p', 'text_o',
             'h2h_wr', 'h2h_total_scaled', 'h2h_sets_margin_scaled']
N_HEAD = len(HEAD_COLS)
print(f'\nN_HEAD scalar features: {N_HEAD} -- {HEAD_COLS}')
assert sym[HEAD_COLS].isna().sum().sum() == 0, 'NaN leaked into head scalar block!'
print('OK -- no NaNs in the head scalar block.')
sym[HEAD_COLS].describe()


## Step 9 — Two-tower shared-weight GRU model + enriched head (Exp C3)

Identical `TwoTowerGRU`-family architecture from Exp C/C2 (`hidden_size=24`, dropout `0.3` on GRU
output + inside MLP head, shared weights across both towers, symmetry guaranteed by the
duplicate-and-flip dataset, `BCEWithLogitsLoss`, AdamW weight decay `1e-4`, `MAX_LEN=24`, early
stopping on chronological `eval`-bucket Brier, seed 42). **Only the head's input width changes:**
`3×24=72` (the GRU embeddings) `+ 15` (the enriched head-scalar block) `= 87`, vs. Exp C2's `74`.

In [ ]:
HIST_T = torch.from_numpy(HIST_ARRAY)
LEN_T = torch.from_numpy(LEN_ARRAY)


class TwoTowerGRUEnriched(nn.Module):
    '''Exp C/C2 TwoTowerGRU architecture, with the enriched N_HEAD-wide scalar block
    concatenated into the MLP head (ELO, CareerWinRate, FormaLast10, MatchCount, Age, TextSkill
    slot -- per player x2 -- plus h2h_win_rate/h2h_total/h2h_sets_margin per matchup).'''

    def __init__(self, input_size=N_FEATS, hidden_size=24, dropout=0.3, mlp_hidden=16, n_head=N_HEAD):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers=1, batch_first=True)
        self.drop_embed = nn.Dropout(dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden_size * 3 + n_head, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1),
        )

    def encode(self, seq, lengths):
        packed = pack_padded_sequence(seq, lengths.clamp(min=1).cpu(), batch_first=True, enforce_sorted=False)
        _, h_n = self.gru(packed)
        return self.drop_embed(h_n[-1])

    def forward(self, player_seq, player_len, opp_seq, opp_len, head_feats):
        h_p = self.encode(player_seq, player_len)
        h_o = self.encode(opp_seq, opp_len)
        combined = torch.cat([h_p, h_o, h_p - h_o, head_feats], dim=1)
        return self.head(combined).squeeze(-1)


def bucket_tensors(bucket_name):
    mask = (sym['Bucket'] == bucket_name).values
    pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
    op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
    y = torch.as_tensor(sym.loc[mask, 'Label'].values, dtype=torch.float32)
    hf = torch.as_tensor(sym.loc[mask, HEAD_COLS].values, dtype=torch.float32)
    return HIST_T[pp], LEN_T[pp], HIST_T[op], LEN_T[op], y, hf

core_pseq, core_plen, core_oseq, core_olen, core_y, core_hf = bucket_tensors('core')
eval_pseq, eval_plen, eval_oseq, eval_olen, eval_y, eval_hf = bucket_tensors('eval')
calib_pseq, calib_plen, calib_oseq, calib_olen, calib_y, calib_hf = bucket_tensors('calib')
test_pseq, test_plen, test_oseq, test_olen, test_y, test_hf = bucket_tensors('test')
print(f'core={core_y.shape[0]:,}  eval={eval_y.shape[0]:,}  calib={calib_y.shape[0]:,}  test={test_y.shape[0]:,}')


## Step 10 — Training loop (identical hyperparameters + early stopping to Exp C/C2)

Manual full-tensor mini-batches (same rationale as `05`: ~21k core rows fit comfortably in
memory), `BATCH_SIZE=512`, `MAX_EPOCHS=60`, `PATIENCE=8`, early stopping on the chronological
`eval` bucket's Brier score (never TEST, never `calib`).

In [ ]:
HIDDEN_SIZE, DROPOUT, WEIGHT_DECAY, LR = 24, 0.3, 1e-4, 2e-3
BATCH_SIZE, MAX_EPOCHS, PATIENCE = 512, 60, 8

torch.manual_seed(RANDOM_STATE)
model = TwoTowerGRUEnriched(hidden_size=HIDDEN_SIZE, dropout=DROPOUT).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
criterion = nn.BCEWithLogitsLoss()
n_core = core_y.shape[0]

def eval_bucket(p_seq, p_len, o_seq, o_len, hf, y):
    model.eval()
    with torch.no_grad():
        logit = model(p_seq, p_len, o_seq, o_len, hf)
    return brier_score_loss(y.numpy(), torch.sigmoid(logit).numpy())

best_brier, best_state, epochs_no_improve = np.inf, None, 0
history, rng = [], np.random.RandomState(RANDOM_STATE)

t0 = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    perm = rng.permutation(n_core)
    total_loss, n_batches = 0.0, 0
    for start in range(0, n_core, BATCH_SIZE):
        idx = perm[start:start + BATCH_SIZE]
        optimizer.zero_grad()
        logit = model(core_pseq[idx], core_plen[idx], core_oseq[idx], core_olen[idx], core_hf[idx])
        loss = criterion(logit, core_y[idx])
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1

    val_brier = eval_bucket(eval_pseq, eval_plen, eval_oseq, eval_olen, eval_hf, eval_y)
    history.append({'epoch': epoch, 'train_loss': total_loss / n_batches, 'val_brier': val_brier})

    improved = val_brier < best_brier - 1e-5
    if improved:
        best_brier, best_state, epochs_no_improve = val_brier, {k: v.clone() for k, v in model.state_dict().items()}, 0
    else:
        epochs_no_improve += 1

    if epoch % 5 == 0 or improved or epoch == 1:
        print(f'epoch {epoch:3d}  train_loss={total_loss/n_batches:.4f}  val_brier={val_brier:.5f}'
              f"{'  <- best' if improved else ''}")
    if epochs_no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch} (no improvement in {PATIENCE} epochs).')
        break

model.load_state_dict(best_state)
print(f'\nTraining done in {time.time()-t0:.1f}s. Best eval-bucket Brier = {best_brier:.5f}')
history_df = pd.DataFrame(history)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history_df['epoch'], history_df['train_loss'], label='train loss (BCE, core)', color='#2563eb')
ax2 = ax.twinx()
ax2.plot(history_df['epoch'], history_df['val_brier'], label='val Brier (eval bucket)', color='#dc2626')
ax.set_xlabel('epoch'); ax.set_ylabel('train BCE loss', color='#2563eb'); ax2.set_ylabel('val Brier', color='#dc2626')
ax.set_title('Exp C3 training curve (enriched GRU) -- early stopping on chronological eval-bucket Brier')
fig.tight_layout()
plt.savefig('../models/exp_c3_training_curve.png', dpi=110)
plt.show()


## Step 11 — Calibration on the `calib` bucket (identical methodology to Exp C/C2)

Temperature scaling vs isotonic regression, both fit on the SAME calib slice; whichever wins on
the calib slice itself is applied to TEST.

In [ ]:
model.eval()
with torch.no_grad():
    calib_logit = model(calib_pseq, calib_plen, calib_oseq, calib_olen, calib_hf)
    test_logit = model(test_pseq, test_plen, test_oseq, test_olen, test_hf)

calib_logit_np, calib_y_np = calib_logit.numpy(), calib_y.numpy()
test_logit_np, test_y_np = test_logit.numpy(), test_y.numpy()

T_param = torch.nn.Parameter(torch.ones(1))
calib_logit_t, calib_y_t = torch.from_numpy(calib_logit_np), torch.from_numpy(calib_y_np)
opt_T = torch.optim.LBFGS([T_param], lr=0.05, max_iter=100)

def _closure():
    opt_T.zero_grad()
    loss = nn.functional.binary_cross_entropy_with_logits(calib_logit_t / T_param, calib_y_t)
    loss.backward()
    return loss

opt_T.step(_closure)
T_value = float(T_param.detach().clamp(min=1e-2))

p_calib_raw = 1 / (1 + np.exp(-calib_logit_np))
p_calib_temp = torch.sigmoid(calib_logit_t / T_value).numpy()

iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(p_calib_raw, calib_y_np)
p_calib_iso = iso.predict(p_calib_raw)

brier_raw = brier_score_loss(calib_y_np, p_calib_raw)
brier_temp = brier_score_loss(calib_y_np, p_calib_temp)
brier_iso = brier_score_loss(calib_y_np, p_calib_iso)
print(f'Fitted temperature T = {T_value:.4f}')
print(f'Calib-slice Brier -- raw: {brier_raw:.5f}, temperature: {brier_temp:.5f}, isotonic: {brier_iso:.5f}')

p_test_raw = 1 / (1 + np.exp(-test_logit_np))
if brier_iso < brier_temp:
    calib_method = 'isotonic'
    p_test_calibrated = iso.predict(p_test_raw)
else:
    calib_method = 'temperature'
    p_test_calibrated = torch.sigmoid(torch.from_numpy(test_logit_np) / T_value).numpy()
print(f'Chosen calibration method for TEST: {calib_method}')


## Step 12 — Score TEST (identical 6,852-row set) — full comparison Exp 0/A/A'/B/B'/C/C2/C3

In [ ]:
def compute_metrics(y_true, p, label):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    return {
        'experiment': label,
        'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
        'brier': brier_score_loss(y_true, p),
        'log_loss': log_loss(y_true, p_c),
        'auc': roc_auc_score(y_true, p),
    }

preds4 = pd.read_parquet(DATA_DIR / 'test_predictions_4way.parquet')
preds_traj = pd.read_parquet(DATA_DIR / 'test_predictions_trajectory.parquet')
preds_c2 = pd.read_parquet(DATA_DIR / 'test_predictions_c2.parquet')

test_bucket_keys = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'PlayerId', 'OpponentId', 'Label']]
merge_check = test_bucket_keys.merge(preds4[['SusretId', 'PlayerId', 'OpponentId', 'Label']],
                                      on=['SusretId', 'PlayerId', 'OpponentId'], suffixes=('_c3', '_exp'))
assert len(merge_check) == len(test_bucket_keys) == len(preds4), 'Row-set mismatch vs Exp 0/A/B/C test set!'
assert (merge_check['Label_c3'] == merge_check['Label_exp']).all(), 'Label mismatch vs Exp 0/A/B/C test set!'
print(f'OK -- Exp C3 test bucket is the exact same {len(preds4):,}-row set as every prior experiment, labels match.')

elo_brier = brier_score_loss(preds4['Label'], preds4['p_exp0_elo'])
comparison = pd.DataFrame([
    compute_metrics(preds4['Label'].values, preds4['p_exp0_elo'].values, 'Exp 0 - ELO baseline'),
    compute_metrics(preds4['Label'].values, preds4['p_expA_calibrated'].values, 'Exp A - ELO-free XGBoost'),
    compute_metrics(preds_traj['Label'].values, preds_traj['p_expAprime_calibrated'].values, "Exp A' - ELO-free XGB + trajectory"),
    compute_metrics(preds4['Label'].values, preds4['p_expB_calibrated'].values, 'Exp B - ELO-included XGBoost'),
    compute_metrics(preds_traj['Label'].values, preds_traj['p_expBprime_calibrated'].values, "Exp B' - ELO-included XGB + trajectory"),
    compute_metrics(preds4['Label'].values, preds4['p_expC_calibrated'].values, 'Exp C - ELO-free GRU'),
    compute_metrics(preds_c2['Label'].values, preds_c2['p_expC2_calibrated'].values, 'Exp C2 - GRU + ELO'),
    compute_metrics(test_y_np, p_test_calibrated, 'Exp C3 - enriched GRU (this notebook)'),
]).set_index('experiment')
comparison['brier_delta_vs_elo_pct'] = (comparison['brier'] - elo_brier) / elo_brier * 100
comparison_display = comparison.round({'accuracy': 4, 'brier': 5, 'log_loss': 5, 'auc': 4, 'brier_delta_vs_elo_pct': 2})
comparison_display


In [ ]:
# Headline point-estimate subset requested by the task: Exp 0 / B' / C / C2 / C3
headline_subset = comparison_display.loc[[
    'Exp 0 - ELO baseline', "Exp B' - ELO-included XGB + trajectory",
    'Exp C - ELO-free GRU', 'Exp C2 - GRU + ELO', 'Exp C3 - enriched GRU (this notebook)',
]]
print('Headline subset (Exp 0 / B\' / C / C2 / C3):')
headline_subset


In [ ]:
print('Headline point-estimate verdict:')
for exp in comparison.index:
    d = comparison.loc[exp, 'brier_delta_vs_elo_pct']
    print(f"  {exp}: Brier {d:+.2f}% vs ELO baseline ({'WORSE' if d > 0 else 'better'} -- higher Brier = worse).")

raw_metrics = compute_metrics(test_y_np, p_test_raw, 'Exp C3 - raw (uncalibrated)')
cal_metrics = compute_metrics(test_y_np, p_test_calibrated, 'Exp C3 - calibrated')
pd.DataFrame([raw_metrics, cal_metrics]).set_index('experiment').round(5)


## Step 13 — Match-level bootstrap CIs + paired-bootstrap Brier differences

Identical methodology to `06`/`08`: 2,000 resamples, seed 42, both perspective rows of a match
always travel together (drawing match IDs with replacement, not individual rows). No walk-forward
CV in this pass, for this notebook.

In [ ]:
preds_all = preds4.merge(preds_traj[['SusretId', 'PlayerId', 'OpponentId', 'p_expAprime_calibrated', 'p_expBprime_calibrated']],
                          on=['SusretId', 'PlayerId', 'OpponentId'], how='inner')
preds_all = preds_all.merge(preds_c2[['SusretId', 'PlayerId', 'OpponentId', 'p_expC2_calibrated']],
                             on=['SusretId', 'PlayerId', 'OpponentId'], how='inner')
exp_c3_preds = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'PlayerId', 'OpponentId']].copy()
exp_c3_preds['p_expC3_calibrated'] = p_test_calibrated
merged = preds_all.merge(exp_c3_preds, on=['SusretId', 'PlayerId', 'OpponentId'], how='inner')
assert len(merged) == len(preds4) == len(preds_traj) == len(preds_c2), 'row-set mismatch across prediction files!'
print(f'Merged all 8 experiments onto {len(merged):,} identical test rows.')

EXPERIMENTS = {
    'Exp 0 - ELO baseline':                   'p_exp0_elo',
    'Exp A - ELO-free XGB':                   'p_expA_calibrated',
    "Exp A' - ELO-free XGB + trajectory":     'p_expAprime_calibrated',
    'Exp B - ELO-included XGB':               'p_expB_calibrated',
    "Exp B' - ELO-included XGB + trajectory": 'p_expBprime_calibrated',
    'Exp C - ELO-free GRU':                   'p_expC_calibrated',
    'Exp C2 - GRU + ELO':                     'p_expC2_calibrated',
    'Exp C3 - enriched GRU':                  'p_expC3_calibrated',
}
y_all = merged['Label'].values
p_all = {name: merged[col].values for name, col in EXPERIMENTS.items()}

def m_dict(y_true, p):
    p_c = np.clip(p, 1e-6, 1 - 1e-6)
    out = {'accuracy': accuracy_score(y_true, (p >= 0.5).astype(int)),
           'brier': brier_score_loss(y_true, p), 'log_loss': log_loss(y_true, p_c)}
    try:
        out['auc'] = roc_auc_score(y_true, p)
    except ValueError:
        out['auc'] = np.nan
    return out

point_estimates = {name: m_dict(y_all, p_all[name]) for name in EXPERIMENTS}

groups = merged.groupby('SusretId').indices
match_ids = np.array(list(groups.keys()))
lens = np.array([len(groups[m]) for m in match_ids])
assert (lens == 2).all(), 'expected exactly 2 rows (both perspectives) per match'
row_idx_2d = np.array([groups[m] for m in match_ids], dtype=np.int64)
n_matches = len(match_ids)
print(f'TEST set: {len(merged):,} rows, {n_matches:,} unique matches.')


In [ ]:
N_BOOT = 2000
rng = np.random.RandomState(RANDOM_STATE)

boot_metrics = {name: {'accuracy': [], 'brier': [], 'log_loss': [], 'auc': []} for name in EXPERIMENTS}
DIFF_KEYS = {
    'C3_minus_C2': ('Exp C3 - enriched GRU', 'Exp C2 - GRU + ELO'),
    'C3_minus_Bprime': ('Exp C3 - enriched GRU', "Exp B' - ELO-included XGB + trajectory"),
    'C3_minus_C': ('Exp C3 - enriched GRU', 'Exp C - ELO-free GRU'),
    'C3_minus_0': ('Exp C3 - enriched GRU', 'Exp 0 - ELO baseline'),
}
boot_diff = {k: [] for k in DIFF_KEYS}

t0 = time.time()
for b in range(N_BOOT):
    sampled_match_pos = rng.randint(0, n_matches, size=n_matches)
    idx = row_idx_2d[sampled_match_pos].reshape(-1)
    y_b = y_all[idx]

    briers = {}
    for name in EXPERIMENTS:
        p_b = p_all[name][idx]
        m = m_dict(y_b, p_b)
        for k in boot_metrics[name]:
            boot_metrics[name][k].append(m[k])
        briers[name] = m['brier']

    for dk, (a, b_) in DIFF_KEYS.items():
        boot_diff[dk].append(briers[a] - briers[b_])

print(f'Match-level bootstrap done: {N_BOOT:,} resamples in {time.time()-t0:.1f}s (seed={RANDOM_STATE})')


### 95% Bootstrap CI table (all 8 experiments)

In [ ]:
rows = []
for name in EXPERIMENTS:
    row = {'experiment': name}
    for metric in ['accuracy', 'brier', 'log_loss', 'auc']:
        arr = np.array(boot_metrics[name][metric])
        lo, hi = np.percentile(arr, [2.5, 97.5])
        row[f'{metric}_point'] = point_estimates[name][metric]
        row[f'{metric}_ci_lo'] = lo
        row[f'{metric}_ci_hi'] = hi
    rows.append(row)

ci_table = pd.DataFrame(rows).set_index('experiment')
ci_table_display = ci_table.round(5)
ci_table_display


### Paired bootstrap on Brier DIFFERENCES (significance) — (C3−C2), (C3−B'), plus (C3−C) and (C3−0) for context

In [ ]:
diff_labels = {
    'C3_minus_C2': "Exp C3 (enriched) minus Exp C2 (GRU+ELO) -- do the enrichments help beyond GRU+ELO?",
    'C3_minus_Bprime': "Exp C3 minus Exp B' (ELO-included XGB + trajectory) -- best GRU vs best-of-everything XGB",
    'C3_minus_C': 'Exp C3 minus Exp C (ELO-free GRU) -- context: does the FULL enrichment bundle (incl. ELO) beat the bare sequence model?',
    'C3_minus_0': 'Exp C3 minus Exp 0 (ELO baseline) -- vs plain ELO',
}
diff_rows = []
for key, label in diff_labels.items():
    arr = np.array(boot_diff[key])
    lo, hi = np.percentile(arr, [2.5, 97.5])
    significant = not (lo < 0 < hi)
    direction = 'lower Brier (BETTER)' if arr.mean() < 0 else 'higher Brier (WORSE)'
    diff_rows.append({'comparison': key, 'description': label, 'mean_diff': arr.mean(),
                       'ci_lo': lo, 'ci_hi': hi, 'significant_95pct': significant, 'direction_of_mean': direction})

diff_table = pd.DataFrame(diff_rows).set_index('comparison')
diff_table_display = diff_table.copy()
for c in ['mean_diff', 'ci_lo', 'ci_hi']:
    diff_table_display[c] = diff_table_display[c].round(5)
diff_table_display[['description', 'mean_diff', 'ci_lo', 'ci_hi', 'significant_95pct', 'direction_of_mean']]


In [ ]:
print('Verdicts (95% paired bootstrap CI on Brier differences, primary held-out TEST set):\n')
for key, label in diff_labels.items():
    r = diff_table.loc[key]
    verdict = 'YES -- statistically significant' if r['significant_95pct'] else 'NOT statistically significant (CI includes 0)'
    print(f"{label}")
    print(f"  mean diff = {r['mean_diff']:+.5f}   95% CI = [{r['ci_lo']:+.5f}, {r['ci_hi']:+.5f}]   -> {verdict}\n")


## Step 14 — Overlaid calibration curves, Exp 0 / B' / C / C2 / C3

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
curves = [
    ('p_exp0_elo', 'Exp 0 - ELO formula', merged, dict(marker='s', linestyle='--', color='#374151')),
    ("p_expBprime_calibrated", "Exp B' - ELO-incl XGB + trajectory", merged, dict(marker='v', linestyle='-', color='#059669')),
    ('p_expC_calibrated', 'Exp C - ELO-free GRU', merged, dict(marker='D', linestyle='-', color='#7c3aed')),
    ('p_expC2_calibrated', 'Exp C2 - GRU + ELO', merged, dict(marker='o', linestyle='-', color='#dc2626')),
]
for col, label, frame, style in curves:
    frac_pos, mean_pred = calibration_curve(frame['Label'].values, frame[col].values, n_bins=10, strategy='uniform')
    ax.plot(mean_pred, frac_pos, label=label, **style)

frac_pos_c3, mean_pred_c3 = calibration_curve(merged['Label'].values, merged['p_expC3_calibrated'].values, n_bins=10, strategy='uniform')
ax.plot(mean_pred_c3, frac_pos_c3, label='Exp C3 - enriched GRU', marker='*', linestyle='-', color='#ea580c', linewidth=2, markersize=10)

ax.plot([0, 1], [0, 1], color='gray', linestyle=':', label='Perfect calibration')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Observed win rate')
ax.set_title("Calibration (Reliability Diagram) -- Exp 0 / B' / C / C2 / C3, TEST set")
ax.legend(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('../models/calibration_curve_c3.png', dpi=120)
plt.show()


## Step 15 — Permutation importance (cheap, no retrain) for every per-step dim + head scalar

Model-agnostic feature importance for the already-trained FULL model: shuffle ONE column across
TEST rows (both towers simultaneously for per-step dims, holding each row's own sequence
length/padding intact), reapply the SAME fitted calibration (`iso`/`T_value` — no refitting), and
measure TEST Brier degradation. Larger degradation = the model relies on that feature more. This
is the cheap complement to the retrain-based ablation in Step 16 below.

In [ ]:
STEP_FEAT_NAMES = ['Won', 'SetMarginScaled', 'DaysRestScaled', 'LigaKoefScaled', 'HomeShareBefore',
                    'OpponentCareerWinRateBefore', 'OppEloBeforeScaled (NEW)', 'WasH2H (NEW)']  # excludes IsPlaceholder (structural, not a "real" signal)

def _apply_calibration(logit):
    p_raw = torch.sigmoid(logit).numpy()
    return iso.predict(p_raw) if calib_method == 'isotonic' else torch.sigmoid(logit / T_value).numpy()

baseline_brier = brier_score_loss(test_y_np, p_test_calibrated)

def permutation_importance_step(feat_idx, n_repeats=3, seed=42):
    rng_local = np.random.RandomState(seed)
    degradations = []
    for _ in range(n_repeats):
        perm_idx = rng_local.permutation(test_pseq.shape[0])
        p_seq_perm, o_seq_perm = test_pseq.clone(), test_oseq.clone()
        p_seq_perm[:, :, feat_idx] = test_pseq[perm_idx, :, feat_idx]
        o_seq_perm[:, :, feat_idx] = test_oseq[perm_idx, :, feat_idx]
        with torch.no_grad():
            logit_perm = model(p_seq_perm, test_plen, o_seq_perm, test_olen, test_hf)
        p_perm = _apply_calibration(logit_perm)
        degradations.append(brier_score_loss(test_y_np, p_perm) - baseline_brier)
    return np.mean(degradations)

def permutation_importance_head(col_idx, n_repeats=3, seed=42):
    rng_local = np.random.RandomState(seed)
    degradations = []
    for _ in range(n_repeats):
        perm_idx = rng_local.permutation(test_hf.shape[0])
        hf_perm = test_hf.clone()
        hf_perm[:, col_idx] = test_hf[perm_idx, col_idx]
        with torch.no_grad():
            logit_perm = model(test_pseq, test_plen, test_oseq, test_olen, hf_perm)
        p_perm = _apply_calibration(logit_perm)
        degradations.append(brier_score_loss(test_y_np, p_perm) - baseline_brier)
    return np.mean(degradations)

step_importance = pd.Series({name: permutation_importance_step(i) for i, name in enumerate(STEP_FEAT_NAMES)})
head_importance = pd.Series({name: permutation_importance_head(i) for i, name in enumerate(HEAD_COLS)})
perm_importance = pd.concat([step_importance, head_importance]).sort_values(ascending=False)
print('Permutation importance (Brier degradation when shuffled -- higher = more important):')
print(perm_importance.to_string())

fig, ax = plt.subplots(figsize=(8, 7))
perm_importance.plot.barh(ax=ax, color=['#ea580c' if ('NEW' in n or n in ('h2h_sets_margin_scaled',)) else '#7c3aed' for n in perm_importance.index])
ax.invert_yaxis()
ax.set_xlabel('Test Brier degradation when shuffled')
ax.set_title('Exp C3 -- per-step + head feature importance (permutation; orange = NEW this notebook)')
plt.tight_layout()
plt.savefig('../models/exp_c3_feature_importance.png', dpi=120)
plt.show()


## Step 16 — Retrain-based ablation: which NEW/enriched blocks actually drive the result?

Unlike Step 15 (single already-trained model, cheap), this retrains the identical architecture /
hyperparameters / seed / data splits with specific input blocks zeroed out — a pure input-masking
ablation, not a re-tuned model (same discipline as `05` Step 15). Runtime-bounded: 4 variants,
~90-100s each (validated in `scripts/exp_c3_validate.py` beforehand).

| Variant | What's zeroed (all else identical to FULL) |
|---|---|
| `FULL` | nothing (reuses the already-trained model from Steps 9-12 above) |
| `NO_STEP_NEW` | the 2 new per-step dims: `OppEloBeforeScaled`, `WasH2H` |
| `NO_HEAD_SETSMARGIN` | only the new `h2h_sets_margin_scaled` head column (the previously-dropped feature) |
| `NO_HEAD_PLAYER_STATS` | the reused per-player head block: `cwr_p/o`, `forma_p/o`, `mc_p/o`, `age_p/o` |
| `NO_HEAD_H2H_REUSED` | the reused matchup head columns `h2h_wr`, `h2h_total_scaled` (sets_margin kept) |


In [ ]:
def run_variant(zero_step_idx, zero_head_idx, label, max_epochs=MAX_EPOCHS, patience=PATIENCE, seed=RANDOM_STATE):
    sf = step_feats.copy()
    for idx in zero_step_idx:
        sf[:, idx] = 0.0

    hist = np.zeros((len(player_log), MAX_LEN, N_FEATS), dtype=np.float32)
    for i in range(len(player_log)):
        gpos = idx_lookup[(igrac_arr[i], susret_arr[i])]
        start = first_idx[igrac_arr[i]]
        p = gpos - start
        if p == 0:
            ph = PLACEHOLDER_VEC.copy()
            for idx in zero_step_idx:
                ph[idx] = 0.0
            hist[i, 0] = ph
        else:
            lo = max(start, gpos - MAX_LEN)
            chunk = sf[lo:gpos]
            hist[i, :chunk.shape[0]] = chunk
    hist_t = torch.from_numpy(hist)

    hf_variant = sym[HEAD_COLS].values.copy()
    for idx in zero_head_idx:
        hf_variant[:, idx] = 0.0

    def _bt(bucket_name):
        mask = (sym['Bucket'] == bucket_name).values
        pp = torch.as_tensor(player_pos[mask], dtype=torch.long)
        op = torch.as_tensor(opp_pos[mask], dtype=torch.long)
        y = torch.as_tensor(sym.loc[mask, 'Label'].values, dtype=torch.float32)
        hf = torch.as_tensor(hf_variant[mask], dtype=torch.float32)
        return hist_t[pp], LEN_T[pp], hist_t[op], LEN_T[op], y, hf

    v_core, v_eval, v_calib, v_test = _bt('core'), _bt('eval'), _bt('calib'), _bt('test')

    torch.manual_seed(seed)
    m = TwoTowerGRUEnriched(hidden_size=HIDDEN_SIZE, dropout=DROPOUT).to(device)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    crit = nn.BCEWithLogitsLoss()
    n_v_core = v_core[4].shape[0]
    rng_local = np.random.RandomState(seed)

    best_brier_v, best_state_v, no_improve = np.inf, None, 0
    for epoch in range(1, max_epochs + 1):
        m.train()
        perm = rng_local.permutation(n_v_core)
        for start in range(0, n_v_core, BATCH_SIZE):
            idx = perm[start:start + BATCH_SIZE]
            opt.zero_grad()
            logit = m(v_core[0][idx], v_core[1][idx], v_core[2][idx], v_core[3][idx], v_core[5][idx])
            loss = crit(logit, v_core[4][idx])
            loss.backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            val_logit = m(v_eval[0], v_eval[1], v_eval[2], v_eval[3], v_eval[5])
        val_brier = brier_score_loss(v_eval[4].numpy(), torch.sigmoid(val_logit).numpy())
        if val_brier < best_brier_v - 1e-5:
            best_brier_v, best_state_v, no_improve = val_brier, {k: v.clone() for k, v in m.state_dict().items()}, 0
        else:
            no_improve += 1
        if no_improve >= patience:
            break
    m.load_state_dict(best_state_v)

    m.eval()
    with torch.no_grad():
        v_calib_logit = m(v_calib[0], v_calib[1], v_calib[2], v_calib[3], v_calib[5])
        v_test_logit = m(v_test[0], v_test[1], v_test[2], v_test[3], v_test[5])
    vc_logit_np, vc_y_np = v_calib_logit.numpy(), v_calib[4].numpy()
    vt_logit_np, vt_y_np = v_test_logit.numpy(), v_test[4].numpy()

    p_vc_raw = 1 / (1 + np.exp(-vc_logit_np))
    iso_v = IsotonicRegression(out_of_bounds='clip')
    iso_v.fit(p_vc_raw, vc_y_np)
    brier_iso_v = brier_score_loss(vc_y_np, iso_v.predict(p_vc_raw))

    T_param_v = torch.nn.Parameter(torch.ones(1))
    vc_logit_t, vc_y_t = torch.from_numpy(vc_logit_np), torch.from_numpy(vc_y_np)
    opt_Tv = torch.optim.LBFGS([T_param_v], lr=0.05, max_iter=100)
    def _closure_v():
        opt_Tv.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(vc_logit_t / T_param_v, vc_y_t)
        loss.backward()
        return loss
    opt_Tv.step(_closure_v)
    T_value_v = float(T_param_v.detach().clamp(min=1e-2))
    p_vc_temp = torch.sigmoid(vc_logit_t / T_value_v).numpy()
    brier_temp_v = brier_score_loss(vc_y_np, p_vc_temp)

    p_vt_raw = 1 / (1 + np.exp(-vt_logit_np))
    if brier_iso_v < brier_temp_v:
        p_vt_final = iso_v.predict(p_vt_raw)
    else:
        p_vt_final = torch.sigmoid(torch.from_numpy(vt_logit_np) / T_value_v).numpy()

    row = compute_metrics(vt_y_np, p_vt_final, label)
    return row

ABLATION_VARIANTS = [
    ('FULL', [], []),
    ('NO_STEP_NEW', [6, 7], []),
    ('NO_HEAD_SETSMARGIN', [], [14]),
    ('NO_HEAD_PLAYER_STATS', [], [2, 3, 4, 5, 6, 7, 8, 9]),
    ('NO_HEAD_H2H_REUSED', [], [12, 13]),
]

ablation_rows = []
t_abl = time.time()
for label, zero_step_idx, zero_head_idx in ABLATION_VARIANTS:
    if label == 'FULL':
        row = compute_metrics(test_y_np, p_test_calibrated, label)  # reuse the already-trained main model
    else:
        row = run_variant(zero_step_idx, zero_head_idx, label)
    row['brier_delta_vs_elo_pct'] = (row['brier'] - elo_brier) / elo_brier * 100
    ablation_rows.append(row)
    print(f"  variant '{label}' done ({time.time()-t_abl:.1f}s elapsed total)")

ablation_table = pd.DataFrame(ablation_rows).set_index('experiment')
ablation_table_display = ablation_table.round({'accuracy': 4, 'brier': 5, 'log_loss': 5, 'auc': 4, 'brier_delta_vs_elo_pct': 2})
ablation_table_display['brier_delta_vs_full'] = (ablation_table['brier'] - ablation_table.loc['FULL', 'brier']).round(5)
ablation_table_display


## Step 17 — Save artifacts

In [ ]:
torch.save({
    'model_state_dict': best_state,
    'hyperparams': {
        'hidden_size': HIDDEN_SIZE, 'dropout': DROPOUT, 'weight_decay': WEIGHT_DECAY,
        'lr': LR, 'batch_size': BATCH_SIZE, 'max_len': MAX_LEN, 'n_feats': N_FEATS,
        'n_head': N_HEAD, 'random_state': RANDOM_STATE,
    },
    'scaling': {
        'setmargin_scale': SETMARGIN_SCALE, 'daysrest_clip': DAYSREST_CLIP, 'ligakoef_scale': LIGAKOEF_SCALE,
        'elo_mean': ELO_MEAN, 'elo_std': ELO_STD, 'age_mean': AGE_MEAN, 'age_std': AGE_STD,
        'matchcount_scale': MATCHCOUNT_SCALE, 'h2htotal_scale': H2HTOTAL_SCALE,
        'h2h_setsmargin_scale': H2H_SETSMARGIN_SCALE, 'text_skill_neutral': TEXT_SKILL_NEUTRAL,
    },
    'step_feature_names': STEP_FEAT_NAMES + ['IsPlaceholder'],
    'head_feature_names': HEAD_COLS,
    'text_skill_slot_note': 'text_p/text_o filled with a neutral constant this run -- see Step 8 markdown for the one-line wiring needed once real Trenutci/Gemini scores are available.',
    'calibration': {'method': calib_method, 'temperature': T_value},
    'best_eval_brier': float(best_brier),
    'test_metrics': compute_metrics(test_y_np, p_test_calibrated, 'Exp C3 - enriched GRU (calibrated)'),
}, MODELS_DIR / 'exp_c3_gru_enriched.pt')

with open(MODELS_DIR / 'exp_c3_isotonic.pkl', 'wb') as f:
    pickle.dump(iso, f)

exp_c3_preds_full = sym.loc[sym['Bucket'] == 'test', ['SusretId', 'Datum', 'SezonaId', 'LigaId', 'PlayerId', 'OpponentId', 'Label', 'Perspective']].copy()
exp_c3_preds_full['p_expC3_raw'] = p_test_raw
exp_c3_preds_full['p_expC3_calibrated'] = p_test_calibrated
exp_c3_preds_full.to_parquet(DATA_DIR / 'test_predictions_c3.parquet', index=False)
merged.to_parquet(DATA_DIR / 'test_predictions_8way.parquet', index=False)

print(f"Saved: {MODELS_DIR / 'exp_c3_gru_enriched.pt'}")
print(f"Saved: {MODELS_DIR / 'exp_c3_isotonic.pkl'}")
print(f"Saved: {DATA_DIR / 'test_predictions_c3.parquet'}")
print(f"Saved: {DATA_DIR / 'test_predictions_8way.parquet'}")


## Step 18 — Written Verdict

**Comparison table (calibrated test Brier, held-out 6,852-row / 3,426-match TEST set, 2025-01-19..2026-07-08):**

| Experiment | Brier | Accuracy | AUC | vs ELO |
|---|---|---|---|---|
| Exp 0 — ELO baseline | 0.17713 | 0.7518 | 0.8210 | — |
| Exp A — ELO-free XGB (no trajectory) | 0.19690 | 0.7018 | 0.7683 | +11.16% (worse) |
| Exp A' — ELO-free XGB + trajectory | 0.17542 | 0.7344 | 0.8172 | −0.97% |
| Exp B — ELO-included XGB | 0.16699 | 0.7520 | 0.8345 | −5.73% |
| Exp B' — ELO-included XGB + trajectory | 0.16545 | 0.7561 | 0.8394 | −6.60% |
| Exp C — ELO-free GRU | 0.16278 | 0.7621 | 0.8411 | −8.11% |
| Exp C2 — GRU + ELO | 0.16088 | 0.7602 | 0.8455 | −9.18% |
| **Exp C3 — enriched GRU (this notebook)** | **0.15869** | **0.7644** | **0.8502** | **−10.41%** |

**Exp C3 is the best model built in this project so far, on every point-estimate metric** — lowest
Brier, highest accuracy, highest AUC of all eight experiments. The enrichment bundle (2 new
per-step dims + 12 per-player head scalars incl. a neutral TextSkill slot + 3 per-matchup head
scalars) moved the point estimate a further ~1.4% relative Brier improvement beyond Exp C2.

**Paired bootstrap verdicts (2,000 match-level resamples, seed 42, primary held-out TEST set):**

| Comparison | Mean Δ Brier | 95% CI | Verdict |
|---|---|---|---|
| C3 − C2 (do the enrichments help beyond GRU+ELO?) | −0.00220 | [−0.00496, +0.00058] | **NOT significant** (CI includes 0) |
| C3 − B' (best GRU vs best-of-everything XGB) | −0.00674 | [−0.01069, −0.00290] | **Significant** |
| C3 − C (full enrichment bundle vs bare ELO-free GRU) | −0.00410 | [−0.00802, −0.00010] | **Significant** (barely — CI upper bound −0.0001) |
| C3 − 0 (vs plain ELO) | −0.01849 | [−0.02278, −0.01428] | **Significant** |

**Honest bottom line: the structural enrichments (without text) do NOT move the needle beyond Exp
C2 in a statistically defensible way.** Exp C3 has the best point estimate of any model in this
project, and it does significantly beat Exp B' and Exp C and Exp 0 — but the specific comparison
this notebook was built to answer (C3 vs C2) lands in the same place as C2 vs C did in `08`: a
real, consistent point-estimate improvement that the paired bootstrap cannot yet distinguish from
noise on this single 3,426-match TEST set. This is the fourth notebook in a row (`06`→`08`→`09`→`10`)
where the *headline* single-split point estimate outruns what the significance test can confirm —
a pattern, not a one-off.

**Which of the new/enriched features actually drove the (non-significant but real) lift?** Two
complementary analyses agree on the same story:

*Permutation importance (no retrain, shuffle-and-measure on the FULL trained model):*

| Feature | Brier degradation when shuffled |
|---|---|
| `elo_p_scaled` / `elo_o_scaled` (Exp C2's ELO-at-head) | 0.0941 / 0.0922 — by far the dominant signal |
| `SetMarginScaled` (per-step, pre-existing) | 0.0494 |
| **`OppEloBeforeScaled` (NEW per-step)** | **0.0214 — 3rd overall, beats even the raw `Won` flag (0.0122)** |
| `LigaKoefScaled` (pre-existing) | 0.0071 |
| `OpponentCareerWinRateBefore` (pre-existing) | 0.0029 |
| `cwr_p` / `h2h_sets_margin_scaled` / `cwr_o` / `age_o` / `age_p` (reused/new head scalars) | 0.0011 – 0.0018 each — small but positive |
| **`WasH2H` (NEW per-step)** | **0.0008 — small but positive** |
| `mc_p/o`, `forma_p/o`, `h2h_total_scaled`, `HomeShareBefore` | ≤ 0.0005 each — negligible |
| `text_p` / `text_o` (neutral-constant SLOT) | ~0 (exactly, as expected — sanity-check passed) |
| `h2h_win_rate` (`h2h_wr`) | slightly **negative** (−0.0001) — not informative to this model once ELO + career stats are present |

*Retrain-based ablation (identical architecture/seed, specific blocks zeroed):*

| Variant | Test Brier | Δ vs FULL |
|---|---|---|
| FULL | 0.15869 | — |
| `NO_STEP_NEW` (drop `OppEloBeforeScaled` + `WasH2H`) | 0.16026 | **+0.00157 (worse)** |
| `NO_HEAD_SETSMARGIN` (drop only the previously-dropped `h2h_sets_margin`) | 0.15776 | −0.00093 (very slightly better — within retrain noise) |
| `NO_HEAD_PLAYER_STATS` (drop CareerWinRate/Forma/MatchCount/Age at head) | 0.15966 | **+0.00097 (worse)** |
| `NO_HEAD_H2H_REUSED` (drop `h2h_win_rate`/`h2h_total`, keep sets margin) | 0.15864 | −0.00005 (no change) |

**Both analyses agree: the single meaningful new contribution is `OpponentEloBefore` as a per-step
sequence feature** — telling the GRU, at every historical step, not just whether the player won
or lost but *how good the opponent was at that moment* — is used more heavily by the model than
the bare win/loss flag itself, and removing it (together with the much smaller `WasH2H`) measurably
hurts test Brier in a real retrain. The reused per-player head block (CareerWinRate/Forma/
MatchCount/Age, first exposed to a GRU here after only ever feeding XGBoost) contributes a smaller
but consistent lift when removed. **The H2H block (`h2h_win_rate`, `h2h_total`, and the specifically
rebuilt-from-scratch `h2h_sets_margin`) contributes essentially nothing once ELO and career-level
signals are already present** — the ablation table shows removing it is a wash or even a very
slight point-estimate improvement, and permutation importance ranks all three H2H-derived scalars
at or near the bottom of the 22-feature list. This is consistent with the project's original
`ELO-substitute` tagging of H2H features in `feature_tags.json` (Phase P2) — with ELO and career
win-rate already in the model, H2H turns out to carry little unique information for this dataset's
match volume (median 0 prior meetings between any given pair — see Step 5's `H2HMeetingsBefore`
describe() output — most pairs simply haven't played each other enough times for an H2H signal to
matter statistically).

**TextSkill slot status: still pending, by design.** `text_p`/`text_o` were wired into the model's
head-scalar tensor as neutral constants (`0.0`) for this run — confirmed inert (exactly 0
permutation importance, as expected for a constant). A concurrent Gemini-scoring pass over
`Trenutci` free-text self-descriptions is running independently of this notebook; this notebook
does not read its output files. Plugging in real per-player scores later requires only replacing
the two constant-fill lines in Step 8 with a merge of the real score onto `sym` by
`PlayerId`/`OpponentId`, then rerunning from Step 9 onward — no architecture change. **This
notebook is therefore "C3 without text"** — whether a real skill-prior signal from self-reported
text moves the needle further (particularly for cold-start / low-match-count players where ELO and
career stats are least informative) remains an open, and now well-scoped, follow-up question.


---

**Update (see `12_walkforward_with_c3_c4.ipynb`):** the "not statistically significant" verdict above for C3 vs C2 was only ever a single-split finding. Walk-forward CV across 4 independent time folds shows a much stronger and more robust picture: Exp C2 finishes third of three GRU variants (behind both Exp C3 and Exp C4) in every single fold, with zero exceptions — the enrichment bundle introduced here (chiefly `OpponentEloBefore`, confirmed the standout feature by both the permutation-importance and retrain ablation above) is what closes the cold-start gap that made Exp B' the outright fold-1 winner in `09_walkforward_top_models.ipynb`. Exp C3 (this notebook, still without text) turns out to be the single best point estimate across the full 4-fold walk-forward too (mean Brier 0.1587±0.0104, the lowest of all 5 models tested in `12`), and the enriched-GRU family (C3 together with Exp C4, `11_enriched_gru_text.ipynb`) occupies the top-2 spots in every one of the 4 folds without exception. So the caution above ("outruns what the significance test can confirm") turns out to have been the right caution to keep at the time, but the underlying enrichment is real and durable — it just took a walk-forward, not a single paired bootstrap on one split, to show it clearly.
